<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-04 14:08:33
=================== PORTFOLIO QUALITY ====================
qualified stocks: 86
with latest results: 86
still star stocks: 46
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.50 C
CAGR/XIRR %:     0.22%
Est FTT Value:   2.74 C
Future Upside:   1.25 C (83.09%)
=================== PERFORMANCE ==========================
Current PnL:             -46.75 L (-24.36%)
Total Booked PnL:        48.59 L (32.6%)
Booked + Current PnL:    1.84 L (1.23%)
CY Booked PnL:           5.71 L (3.81%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               1.57 L (1.06%)
CY Investment:            1.92 C
Reserve:                  1.90 K
=================== P&L DETAIL ============================
Total Profit:             1.93 L
Total Loss:               -48.68 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,-2.87,27.07,6.64,35.51,17683.0,56731.0,266305.0,1285.01,25.76,58.0,H-LC,11.66,120.0,3.21,1.78,40.14,AR,ATH,PHARMA
76,TTKPRESTIG,4.84,-28.47,39.94,0.09,28790.0,-28694.0,72083.0,770.00,-12.93,59.0,M-SC,4.79,250.0,-1.00,0.48,29.74,OX40N,NTT,DURABLES
12,BOSCH-HCIL,0.52,-24.44,32.34,-0.01,31229.0,-31239.0,96566.0,2282.00,754.44,47.0,M-SC,20.88,231.0,-1.00,0.64,64.37,OX40N,BTT,AC
66,SIS,1.63,-11.50,37.02,21.26,36377.0,-12768.0,98264.0,528.00,89.02,45.0,H-SC,8.42,168.0,-0.35,0.66,46.10,OX40N,NTT,MISC
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
52,MARUTI,0.48,-12.87,43.55,25.08,114263.0,-38746.0,262372.0,17120.06,-18.15,57.0,X-LC,14.22,33.0,-0.34,1.75,6.77,XR,ATH,AUTO
38,HYUNDAI,1.53,6.60,35.02,43.94,93829.0,16598.0,267930.0,2782.73,10.35,60.0,X-LC,6.24,29.0,0.18,1.79,23.02,X40N,ATH,AUTO
51,M&M,0.72,-8.55,28.47,17.48,78385.0,-25749.0,275326.0,3762.88,-20.31,52.0,X-LC,9.93,25.0,-0.33,1.84,6.46,X40N,ATH,AUTO
35,HDFCLIFE,0.50,-8.51,53.63,40.56,152670.0,-26486.0,284673.0,811.41,-33.38,47.0,X-LC,10.25,31.0,-0.17,1.90,2.83,X40,ATH,INSURANCE
4,ASIANPAINT,3.46,-12.08,41.45,24.36,120634.0,-39996.0,291036.0,3430.49,-22.98,58.0,X-LC,6.83,30.0,-0.33,1.94,15.32,X40,ATH,PAINTS


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
29,GREENPANEL,8.72,-63.92,296.75,43.14,196006.0,-117027.0,66051.0,537.00,NaN,41.0,NaN,24.49,225.0,-0.60,0.44,8.71,XY24,NTT,MISC
49,KANSAINER,8.61,-34.80,74.05,13.47,130191.0,-93852.0,175815.0,340.00,-31.79,62.0,H-SC,8.45,162.0,-0.72,1.17,24.27,XY24,NTT,PAINTS
26,GILLETTE,6.39,-11.51,45.43,28.69,102068.0,-29226.0,224670.0,10891.01,-36.19,62.0,X-SC,6.69,83.0,-0.29,1.50,8.55,X40,ATH,FMCG
16,CERA,5.51,-37.96,99.70,23.89,108799.0,-66777.0,109126.0,9475.00,NaN,43.0,NaN,11.45,153.0,-0.61,0.73,7.68,OX40N,NTT,CERAMICS
62,REPCOHOME,5.11,-29.44,169.03,89.82,341691.0,-84357.0,202148.0,880.00,NaN,50.0,NaN,13.95,161.0,-0.25,1.35,7.72,XY24,NTT,FINANCE


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
3,ALKYLAMINE,-3.70,-32.29,184.37,92.54,126037.0,-32602.0,68361.0,4520.85,NaN,35.0,NaN,9.78,142.0,-0.26,0.46,29.81,SR,ATH,CHEMICALS
85,ZYDUSLIFE,-2.87,27.07,6.64,35.51,17683.0,56731.0,266305.0,1285.01,25.76,58.0,H-LC,11.66,120.0,3.21,1.78,40.14,AR,ATH,PHARMA
72,TCS,-2.81,-26.85,73.27,26.75,261599.0,-131041.0,357034.0,4208.34,-40.96,57.0,X-LC,4.96,2.0,-0.50,2.38,23.83,X40,ATH,IT
45,INFY,-2.63,-19.13,76.40,42.66,247682.0,-76664.0,324191.0,1972.00,-41.25,55.0,X-LC,8.98,3.0,-0.31,2.16,16.06,X40,NTT,IT
84,WIPRO,-2.52,-11.30,117.59,93.01,261018.0,-28267.0,221973.0,420.00,-25.10,64.0,M-LC,8.70,98.0,-0.11,1.48,23.11,XR,NTT,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.85,51.05,36.67,106.44,81780.0,75372.0,223016.0,1629.94,10.09,42.0,M-SC,26.32,220.0,0.92,1.49,68.98,OX40N,ATH,CABLES
66,SIS,1.63,-11.50,37.02,21.26,36377.0,-12768.0,98264.0,528.00,89.02,45.0,H-SC,8.42,168.0,-0.35,0.66,46.10,OX40N,NTT,MISC
12,BOSCH-HCIL,0.52,-24.44,32.34,-0.01,31229.0,-31239.0,96566.0,2282.00,754.44,47.0,M-SC,20.88,231.0,-1.00,0.64,64.37,OX40N,BTT,AC
76,TTKPRESTIG,4.84,-28.47,39.94,0.09,28790.0,-28694.0,72083.0,770.00,-12.93,59.0,M-SC,4.79,250.0,-1.00,0.48,29.74,OX40N,NTT,DURABLES
49,KANSAINER,8.61,-34.80,74.05,13.47,130191.0,-93852.0,175815.0,340.00,-31.79,62.0,H-SC,8.45,162.0,-0.72,1.17,24.27,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.85,51.05,36.67,106.44,81780.0,75372.0,223016.0,1629.94,10.09,42.0,M-SC,26.32,220.0,0.92,1.49,68.98,OX40N,ATH,CABLES
85,ZYDUSLIFE,-2.87,27.07,6.64,35.51,17683.0,56731.0,266305.0,1285.01,25.76,58.0,H-LC,11.66,120.0,3.21,1.78,40.14,AR,ATH,PHARMA


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.85,51.05,36.67,106.44,81780.0,75372.0,223016.0,1629.94,10.09,42.0,M-SC,26.32,220.0,0.92,1.49,68.98,OX40N,ATH,CABLES
67,STARHEALTH,1.51,2.65,43.46,47.26,125410.0,7442.0,288565.0,761.00,-9.42,46.0,H-SC,5.19,173.0,0.06,1.93,24.77,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,4.85,51.05,36.67,106.44,81780.0,75372.0,223016.0,1629.94,10.09,42.0,M-SC,26.32,220.0,0.92,1.49,68.98,OX40N,ATH,CABLES
80,VALIANTORG,1.74,-8.90,91.17,74.15,135879.0,-14566.0,149039.0,838.00,NaN,52.0,NaN,49.84,145.0,-0.11,0.99,121.79,XR,NTT,CHEMICALS
84,WIPRO,-2.52,-11.30,117.59,93.01,261018.0,-28267.0,221973.0,420.00,-25.10,64.0,M-LC,8.70,98.0,-0.11,1.48,23.11,XR,NTT,IT
70,TANLA,1.84,-25.71,189.64,115.17,381459.0,-69612.0,201149.0,1903.93,NaN,79.0,NaN,9.67,147.0,-0.18,1.34,77.82,AR,ATH,IT
42,INDIAMART,2.38,-32.19,196.30,100.92,164174.0,-39702.0,83634.0,4675.60,NaN,48.0,NaN,16.80,140.0,-0.24,0.56,6.11,AR,ATH,MISC


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE
6,AWL,-0.36,-43.63,196.27,67.02,394228.0,-155449.0,200860.0,485.00,-58.47,31.0,X-SC,6.98,81.0,-0.39,1.34,0.00,XY24,NTT,FMCG
37,HONAUT,2.31,-22.51,94.72,50.88,141535.0,-43413.0,149425.0,58192.80,-39.50,31.0,X-SC,5.86,90.0,-0.31,1.00,13.84,X40N,ATH,ELECTRICAL
8,BAJAJHLDNG,0.55,-6.78,36.22,26.99,105655.0,-21223.0,291704.0,14191.91,-15.21,42.0,X-LC,3.99,28.0,-0.20,1.95,21.30,X40,ATH,FINANCE
27,GLAXO,-0.18,1.44,33.77,35.69,68267.0,2868.0,202152.0,3380.16,-23.91,43.0,X-MC,10.42,56.0,0.04,1.35,20.15,X40,ATH,PHARMA


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
55,PAGEIND,0.03,-7.38,43.44,32.85,107979.0,-19809.0,248570.0,50936.08,-31.08,50.0,X-MC,1.30,55.0,-0.18,1.66,18.70,X40,ATH,APPARELS
1,ABBOTINDIA,1.70,-11.91,30.96,15.37,48781.0,-21295.0,157560.0,34390.26,-28.20,53.0,X-MC,1.56,58.0,-0.44,1.05,6.73,X40,ATH,PHARMA
17,CIPLA,-0.12,4.88,18.13,23.89,57167.0,14660.0,315315.0,1648.14,18.90,63.0,X-LC,1.94,18.0,0.26,2.10,18.11,X40N,ATH,PHARMA
11,BERGEPAINT,3.12,-10.95,40.73,25.32,93483.0,-28229.0,229520.0,680.00,-22.61,60.0,X-MC,2.31,68.0,-0.30,1.53,20.36,XY24,NTT,PAINTS
33,HCLTECH,-2.15,-3.99,30.93,25.70,107193.0,-14407.0,346566.0,1837.07,-8.41,62.0,X-LC,3.88,8.0,-0.13,2.31,38.27,X40,ATH,IT


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.36,-43.63,196.27,67.02,394228.0,-155449.0,200860.0,485.00,-58.47,31.0,X-SC,6.98,81.0,-0.39,1.34,0.00,XY24,NTT,FMCG
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE
35,HDFCLIFE,0.50,-8.51,53.63,40.56,152670.0,-26486.0,284673.0,811.41,-33.38,47.0,X-LC,10.25,31.0,-0.17,1.90,2.83,X40,ATH,INSURANCE
28,GODREJCP,2.80,-22.85,71.23,32.10,137131.0,-57018.0,192519.0,1452.16,-24.23,50.0,X-MC,16.81,32.0,-0.42,1.28,4.31,X40N,ATH,FMCG
19,COLPAL,4.37,-37.58,120.71,37.76,198423.0,-98985.0,164380.0,3628.04,-24.34,44.0,X-MC,14.06,62.0,-0.50,1.10,4.53,XY25,ATH,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,-1.20,-47.76,150.78,31.02,168198.0,-101968.0,111552.0,17484.40,-61.63,47.0,X-SC,24.65,57.0,-0.61,0.74,5.02,X40,ATH,FMCG
40,ICICIPRULI,0.66,-32.20,27.50,-13.56,36730.0,-63443.0,133562.0,522.35,-56.58,27.0,X-MC,9.01,78.0,-1.73,0.89,0.66,X40,ATH,INSURANCE
2,ACC,1.62,-46.78,228.48,74.81,326023.0,-125436.0,142692.0,3906.00,-47.20,53.0,X-SC,16.33,85.0,-0.38,0.95,5.71,XY24,BTT,CEMENT
37,HONAUT,2.31,-22.51,94.72,50.88,141535.0,-43413.0,149425.0,58192.80,-39.50,31.0,X-SC,5.86,90.0,-0.31,1.00,13.84,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,1.70,-11.91,30.96,15.37,48781.0,-21295.0,157560.0,34390.26,-28.20,53.0,X-MC,1.56,58.0,-0.44,1.05,6.73,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,2.12,-20.36,61.60,28.70,214680.0,-89089.0,348506.0,452.00,-27.90,67.0,X-LC,8.28,1.0,-0.41,2.33,9.69,X40,NTT,FMCG
72,TCS,-2.81,-26.85,73.27,26.75,261599.0,-131041.0,357034.0,4208.34,-40.96,57.0,X-LC,4.96,2.0,-0.50,2.38,23.83,X40,ATH,IT
45,INFY,-2.63,-19.13,76.40,42.66,247682.0,-76664.0,324191.0,1972.00,-41.25,55.0,X-LC,8.98,3.0,-0.31,2.16,16.06,X40,NTT,IT
74,TMPV,1.14,-31.88,114.40,46.05,348642.0,-142615.0,304757.0,600.00,-85.99,51.0,X-LC,17.65,4.0,-0.41,2.03,7.14,XY24,NTT,AUTO
33,HCLTECH,-2.15,-3.99,30.93,25.70,107193.0,-14407.0,346566.0,1837.07,-8.41,62.0,X-LC,3.88,8.0,-0.13,2.31,38.27,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,0.52,-24.44,32.34,-0.01,31229.0,-31239.0,96566.0,2282.00,754.44,47.0,M-SC,20.88,231.0,-1.00,0.64,64.37,OX40N,BTT,AC
80,VALIANTORG,1.74,-8.90,91.17,74.15,135879.0,-14566.0,149039.0,838.00,NaN,52.0,NaN,49.84,145.0,-0.11,0.99,121.79,XR,NTT,CHEMICALS
70,TANLA,1.84,-25.71,189.64,115.17,381459.0,-69612.0,201149.0,1903.93,NaN,79.0,NaN,9.67,147.0,-0.18,1.34,77.82,AR,ATH,IT
25,FINCABLES,4.85,51.05,36.67,106.44,81780.0,75372.0,223016.0,1629.94,10.09,42.0,M-SC,26.32,220.0,0.92,1.49,68.98,OX40N,ATH,CABLES
85,ZYDUSLIFE,-2.87,27.07,6.64,35.51,17683.0,56731.0,266305.0,1285.01,25.76,58.0,H-LC,11.66,120.0,3.21,1.78,40.14,AR,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,1.84,-25.71,189.64,115.17,381459.0,-69612.0,201149.0,1903.93,NaN,79.0,NaN,9.67,147.0,-0.18,1.34,77.82,AR,ATH,IT
71,TATAELXSI,-0.09,-46.48,146.71,32.03,103508.0,-61282.0,70553.0,9161.00,-39.28,58.0,H-SC,14.12,158.0,-0.59,0.47,23.94,OX40N,NTT,IT
84,WIPRO,-2.52,-11.30,117.59,93.01,261018.0,-28267.0,221973.0,420.00,-25.10,64.0,M-LC,8.70,98.0,-0.11,1.48,23.11,XR,NTT,IT
33,HCLTECH,-2.15,-3.99,30.93,25.70,107193.0,-14407.0,346566.0,1837.07,-8.41,62.0,X-LC,3.88,8.0,-0.13,2.31,38.27,X40,ATH,IT
72,TCS,-2.81,-26.85,73.27,26.75,261599.0,-131041.0,357034.0,4208.34,-40.96,57.0,X-LC,4.96,2.0,-0.50,2.38,23.83,X40,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.11
1,25,49.33
2,50,80.37


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    42.53
MC    20.65
SC    20.12
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.35
XY24     22.28
X40N     18.59
XR        9.82
XY25      9.51
OX40N     6.63
AR        5.04
SR        0.76
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    39.27
X-MC    18.29
H-SC     8.03
X-SC     7.05
M-SC     5.04
H-LC     1.78
M-LC     1.48
M-MC     1.05
H-MC     0.98
L-MC     0.33
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.90,-27.98,73.94
IT,12.86,-32.90,102.35
FINANCE,10.44,-28.85,83.89
AUTO,8.29,-20.58,63.60
PHARMA,6.28,5.63,20.39
INSURANCE,5.84,-13.13,43.90
MISC,5.43,-51.79,112.80
PAINTS,5.36,-28.54,51.22
BANKS,4.94,-21.23,82.03


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4095375.0,19
X40,2293059.0,17
XR,1581862.0,13
X40N,1552753.0,12
AR,1039203.0,7
OX40N,797479.0,10
XY25,790928.0,6
SR,299902.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,3058630.0,19
X-MC,1666956.0,14
X-SC,1217634.0,6
H-SC,947505.0,8
M-SC,406605.0,6
H-MC,267124.0,1
M-LC,261018.0,1
M-MC,88057.0,1
L-MC,53615.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1364172.0      8
X-SC       XY24       720251.0      2
X-LC       X40N       605880.0      5
           XY24       588109.0      2
H-SC       XY24       580773.0      3
X-MC       X40N       552886.0      4
           X40        537376.0      6
X-LC       XY25       386206.0      3
X-MC       XY25       316665.0      2
X-SC       X40        270266.0      2
H-MC       XY24       267124.0      1
M-LC       XR         261018.0      1
X-MC       XY24       260029.0      2
H-SC       OX40N      251082.0      3
X-SC       X40N       227117.0      2
M-SC       OX40N      209140.0      4
           XY24       137863.0      1
H-SC       XR         115650.0      2
X-LC       XR         114263.0      1
M-MC       XY25        88057.0      1
M-SC       XR          59602.0      1
L-MC       XR          53615.0      1
H-LC       AR          17683.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 28.0 seconds
